# Studio 1 — A cell-by-cell scRNA-seq workflow

This notebook is deliberately organized as **one small action per code cell**. Run a cell, look at its result, read the next explanation, and then make the next decision.

> **Scientific order:** raw counts → QC evidence → QC cutoffs → doublet evidence → doublet decision → normalization → log transformation → PCA → neighbors/UMAP → clustering. Doublets are assessed before normalization and embeddings because Scrublet uses raw counts and doublets can distort later analysis.

The notebook supports one or two Cell Ranger `.h5` files or `.h5ad` objects. It does not assign cell types, correct batches, or prove biological mechanisms.

## Progress checklist

- [ ] Load data and preserve raw counts.
- [ ] Calculate QC metrics for genes, counts, mitochondria, ribosomal proteins, and hemoglobin.
- [ ] Inspect each plot **before** entering the corresponding cutoff.
- [ ] Filter cells and genes with documented choices.
- [ ] Score and review doublets before choosing whether to remove them.
- [ ] Normalize, log-transform, select HVGs, and run PCA.
- [ ] Build a graph, calculate UMAP, and compare three clustering methods.
- [ ] Save a reproducible `.h5ad` object with every decision recorded.

## Workflow map

```text
raw count matrix
  → preserve raw counts
  → inspect QC plots
  → enter one QC cutoff at a time
  → filter
  → Scrublet score plot → doublet decision
  → size-factor normalization → log1p
  → highly variable genes → scale → PCA
  → choose PCs → neighbor graph → UMAP
  → Louvain → Leiden → k-means
  → save an .h5ad file with parameters
```

**Important:** There are no universal cutoffs. Every threshold below is a starting point that must be checked against your tissue, chemistry, and QC distributions.

## Step 1 — Import packages

In [ ]:
# Import the packages used throughout the notebook.
from pathlib import Path
from importlib.metadata import version

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from scipy import sparse
from sklearn.cluster import KMeans

# Set a reproducible seed and readable notebook figure settings.
RANDOM_SEED = 0
sc.settings.set_figure_params(dpi=100, facecolor="white")

In [ ]:
# Record software versions before analysis begins.
for package in ["scanpy", "anndata", "numpy", "pandas", "scrublet", "igraph", "leidenalg", "scikit-learn"]:
    print(f"{package}: {version(package)}")

## Step 2 — Tell the notebook where the count matrices are

In [ ]:
# Replace these example paths with approved data files.
# Use one entry for one sample or two entries for the Studio 1 two-sample activity.
INPUT_FILES = {
    "A054": Path("data/10x_A054_cellranger/filtered_feature_bc_matrix.h5"),
    "A055": Path("data/10x_A055_cellranger/filtered_feature_bc_matrix.h5"),
}

In [ ]:
# Choose the file name for the final reproducible analysis object.
OUTPUT_PATH = Path("results/studio1_preprocessed_clusters.h5ad")

## Step 3 — Load the files

In [ ]:
path = "/Users/terooatt/Documents/Bio559r/Data/"
data1 = sc.read_10x_h5(path + "10x_A054_cellranger/filtered_feature_bc_matrix.h5")
data2 = sc.read_10x_h5(path + "10x_A055_cellranger/filtered_feature_bc_matrix.h5")

data1.var_names_make_unique()
data2.var_names_make_unique()

In [ ]:
# Combine samples and save their identity in adata.obs['sample'].
# combine adata
import anndata as ad

adata = ad.concat(
    [data1, data2],
    join="outer",
    label="sample",
    keys=["data1", "data2"],
    index_unique="-"
)

# Use a sparse matrix when possible because scRNA-seq counts are mostly zero.
if not sparse.issparse(adata.X):
    adata.X = sparse.csr_matrix(adata.X)
print(adata)

In [ ]:
# Check how many cells came from each sample before filtering.
print(adata.obs["sample"].value_counts().sort_index())

## Step 4 — Confirm raw counts and preserve them

In [ ]:
# Check for values inconsistent with non-negative integer-like UMI counts.
nonzero = adata.X.data if sparse.issparse(adata.X) else np.asarray(adata.X).ravel()
if np.any(nonzero < 0):
    raise ValueError("Negative values found. This notebook expects raw non-negative counts.")
if np.allclose(nonzero, np.round(nonzero)):
    print("Values are integer-like, consistent with a raw count matrix.")
else:
    print("WARNING: Values are not integer-like. Confirm that this is raw counts before continuing.")

In [ ]:
# Save raw counts once. Later transformations must not overwrite this layer.
adata.layers["counts"] = adata.X.copy()
print("Saved raw counts in adata.layers['counts'].")

## Step 5 — Define QC gene groups and calculate QC metrics

Mitochondrial (`MT-`), ribosomal-protein (`RPS`/`RPL`), and hemoglobin (`HBA`/`HBB`) prefixes are made case-insensitive here. **Ribosomal-protein genes are not the same as rRNA transcripts.** Check that these labels match the organism and reference annotation used for your data.

In [ ]:
# Mark mitochondrial genes; this works for common human and mouse-style gene symbols.
gene_upper = adata.var_names.str.upper()
adata.var["mt"] = gene_upper.str.startswith("MT-")
print("Mitochondrial genes found:", int(adata.var["mt"].sum()))

In [ ]:
# Mark ribosomal-protein genes.
adata.var["ribo"] = gene_upper.str.startswith(("RPS", "RPL"))
print("Ribosomal-protein genes found:", int(adata.var["ribo"].sum()))

In [ ]:
# Mark hemoglobin genes.
adata.var["hb"] = gene_upper.str.match(r"^HB[AB]")
print("Hemoglobin genes found:", int(adata.var["hb"].sum()))

In [ ]:
# Calculate per-cell QC metrics from raw counts and add them to adata.obs.
sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt", "ribo", "hb"],
    percent_top=None,
    log1p=False,
    inplace=True,
)
print("Created QC columns, including pct_counts_mt, pct_counts_ribo, and pct_counts_hb.")

## Step 6 — Inspect gene-count QC, then choose gene-count cutoffs

In [ ]:
# Look at how many genes each cell expresses. Run this BEFORE choosing gene-count cutoffs.
sc.pl.violin(adata, keys="n_genes_by_counts", groupby="sample", jitter=0.35, size=2)

### Decision 6A — Minimum genes per cell

After inspecting the violin plot, choose a lower cutoff that removes nearly empty droplets while retaining plausible low-RNA cells for this tissue.

In [ ]:
# Edit this value only after inspecting the previous plot, then record your rationale.
MIN_GENES_PER_CELL = 200
print("Minimum genes per cell:", MIN_GENES_PER_CELL)

### Decision 6B — Maximum genes per cell

Use the same gene-count distribution to choose an upper cutoff. Very high values can indicate doublets, but can also occur in genuine high-RNA cell types.

In [ ]:
# Edit this value only after inspecting the gene-count plot.
MAX_GENES_PER_CELL = 5000
print("Maximum genes per cell:", MAX_GENES_PER_CELL)

## Step 7 — Inspect mitochondrial percentage, then choose its cutoff

In [ ]:
# Examine mitochondrial percentages before entering a mitochondrial cutoff.
sc.pl.violin(adata, keys="pct_counts_mt", groupby="sample", jitter=0.35)
sc.pl.scatter(adata, x="pct_counts_mt", y="total_counts", color="sample")

### Decision 7 — Mitochondrial percentage cutoff

High mitochondrial percentage may indicate low-quality or stressed cells, but its interpretation varies by tissue and protocol. Choose a value after inspecting the plot.

In [ ]:
# Edit this value only after inspecting mitochondrial QC plots.
MAX_MITO_PERCENT = 35
print("Maximum mitochondrial percent:", MAX_MITO_PERCENT)

## Step 8 — Inspect ribosomal-protein percentage, then decide whether to filter

In [ ]:
# Examine ribosomal-protein percentage before deciding whether it should be a filter.
sc.pl.violin(adata, keys="pct_counts_ribo", groupby="sample", jitter=0.35)

### Decision 8 — Ribosomal-protein percentage cutoff

Ribosomal-protein signal can be biological. Leave this as `None` to keep all cells on this metric, or enter a justified percentage after inspecting the plot.

In [ ]:
# Use None for no ribosomal-protein filter, or enter a percentage after inspecting the plot.
MAX_RIBO_PERCENT = None
print("Maximum ribosomal-protein percent:", MAX_RIBO_PERCENT)

## Step 9 — Inspect hemoglobin percentage, then decide whether to filter

In [ ]:
# Examine hemoglobin percentage before deciding whether it should be a filter.
sc.pl.violin(adata, keys="pct_counts_hb", groupby="sample", jitter=0.35)

### Decision 9 — Hemoglobin percentage cutoff

Hemoglobin signal may represent ambient RNA, erythroid cells, or a relevant biological population. Leave this as `None` unless the plot and experimental context justify a cutoff.

In [ ]:
# Use None for no hemoglobin filter, or enter a percentage after inspecting the plot.
MAX_HB_PERCENT = None
print("Maximum hemoglobin percent:", MAX_HB_PERCENT)

## Step 10 — Inspect raw UMI totals, then choose an optional total-count cutoff

In [ ]:
# Look at raw UMI totals. This can support a high-count cutoff decision but does not prove doublets.
sc.pl.violin(adata, keys="total_counts", groupby="sample", jitter=0.35)
sc.pl.scatter(adata, x="total_counts", y="n_genes_by_counts", color="sample")

### Decision 10 — Optional maximum UMI-count cutoff

Use `None` to avoid filtering by total UMI count. If you enter a number, justify it from the plot and biology; do not remove a real high-RNA population merely because it is high-count.

In [ ]:
# Use None for no UMI-count filter, or enter a number after inspecting the plots.
MAX_TOTAL_COUNTS = None
print("Maximum raw UMI counts:", MAX_TOTAL_COUNTS)

## Step 11 — Apply the QC choices you made above

In [ ]:
# Start a pass/fail table using the gene-count and mitochondrial decisions.
adata.obs["pass_min_genes"] = adata.obs["n_genes_by_counts"] >= 200
adata.obs["pass_max_genes"] = adata.obs["n_genes_by_counts"] <= 5000
adata.obs["pass_mito"] = adata.obs["pct_counts_mt"] <= 20

In [ ]:
# Apply the optional ribosomal-protein decision only when a cutoff was entered.
adata.obs["pass_ribo"] = True if MAX_RIBO_PERCENT is None else adata.obs["pct_counts_ribo"] <= 40

In [ ]:
# Apply the optional hemoglobin decision only when a cutoff was entered.
adata.obs["pass_hb"] = True if MAX_HB_PERCENT is None else adata.obs["pct_counts_hb"] <= 0.005

In [ ]:
# Apply the optional total-count decision only when a cutoff was entered.
adata.obs["pass_total_counts"] = True if MAX_TOTAL_COUNTS is None else adata.obs["total_counts"] <= 20000

In [ ]:
# Combine all QC decisions. Inspect how many cells pass before subsetting.
qc_rules = ["pass_min_genes", "pass_max_genes", "pass_mito", "pass_ribo", "pass_hb", "pass_total_counts"]
adata.obs["pass_qc"] = adata.obs[qc_rules].all(axis=1)
print(adata.obs["pass_qc"].value_counts())

In [ ]:
# Keep only cells that passed every documented QC rule.
adata = adata[adata.obs["pass_qc"]].copy()
print(f"Cells remaining after cell QC: {adata.n_obs:,}")

## Step 12 — Choose a gene-detection cutoff, then filter genes

In [ ]:
# Decide how many retained cells must express a gene. This removes very rare/noisy features.
MIN_CELLS_PER_GENE = 3
print("Minimum cells per gene:", MIN_CELLS_PER_GENE)

In [ ]:
# Remove genes detected in fewer than the chosen number of retained cells.
sc.pp.filter_genes(adata, min_cells=MIN_CELLS_PER_GENE)
print(f"After gene filtering: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

## Step 13 — Choose an expected doublet rate, then run Scrublet

Scrublet must see raw counts, which are still in `adata.X` at this stage. With two samples, it scores each sample separately using `batch_key="sample"`.

In [ ]:
# Choose an expected doublet rate using the platform loading/recovery information.
EXPECTED_DOUBLET_RATE = 0.06
print("Expected doublet rate:", EXPECTED_DOUBLET_RATE)

In [ ]:
# Score doublet candidates from raw counts.
scrublet_kwargs = {"expected_doublet_rate": EXPECTED_DOUBLET_RATE, "random_state": RANDOM_SEED}
if adata.obs["sample"].nunique() > 1:
    scrublet_kwargs["batch_key"] = "sample"
sc.pp.scrublet(adata, **scrublet_kwargs)
print(adata.obs[["doublet_score", "predicted_doublet"]].describe(include="all"))

## Step 14 — Inspect doublet scores, then choose a doublet rule

In [ ]:
# Inspect score distributions before defining doublet candidates.
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(data=adata.obs, x="doublet_score", hue="sample", bins=150, element="step", stat="density", common_norm=False, ax=ax)
ax.set_title("Scrublet doublet-score distribution")
plt.show()

In [ ]:
# Compare doublet scores with total counts. This is evidence, not automatic proof of a doublet.
sc.pl.scatter(adata, x="total_counts", y="doublet_score", color="predicted_doublet")

In [ ]:
adata.obs

### Decision 14 — Doublet rule

Use Scrublet's predicted label, or enter a score cutoff after reviewing both plots and the expected doublet rate. Record the evidence used.

In [ ]:
# Use predicted_doublet by default. Set this to False only when using a custom score threshold.
USE_SCRUBLET_PREDICTIONS = True

In [ ]:
# Use None to keep Scrublet's prediction, or enter a justified score threshold such as 0.25.
CUSTOM_DOUBLET_SCORE_THRESHOLD = 0.2

In [ ]:
# Create the candidate label from the selected doublet rule.
if CUSTOM_DOUBLET_SCORE_THRESHOLD is None:
    if not USE_SCRUBLET_PREDICTIONS:
        raise ValueError("Enable Scrublet predictions or enter CUSTOM_DOUBLET_SCORE_THRESHOLD.")
    adata.obs["doublet_candidate"] = adata.obs["predicted_doublet"].astype(bool)
    doublet_rule = "Scrublet predicted_doublet"
else:
    adata.obs["doublet_candidate"] = adata.obs["doublet_score"] >= CUSTOM_DOUBLET_SCORE_THRESHOLD
    doublet_rule = f"doublet_score >= {CUSTOM_DOUBLET_SCORE_THRESHOLD}"
print(doublet_rule)
print(adata.obs["doublet_candidate"].value_counts())

## Step 15 — Decide whether to remove the reviewed doublet candidates

In [ ]:
# Keep this False during teaching/review. Change to True only after documenting the doublet decision.
REMOVE_DOUBLET_CANDIDATES = False
print("Remove doublet candidates:", REMOVE_DOUBLET_CANDIDATES)

In [ ]:
# Apply the reviewed doublet-removal decision.
if REMOVE_DOUBLET_CANDIDATES:
    adata = adata[~adata.obs["doublet_candidate"]].copy()
print(f"Cells remaining after doublet decision: {adata.n_obs:,}")

## Step 16 — Choose a normalization target, then calculate size factors

In [ ]:
# Choose the total count target for per-cell normalization.
TARGET_SUM = 10_000
print("Normalization target sum:", TARGET_SUM)

In [ ]:
# Calculate each cell's raw library size and explicit size factor before normalization.
counts_per_cell = np.asarray(adata.layers["counts"].sum(axis=1)).ravel()
if np.any(counts_per_cell == 0):
    raise ValueError("Zero-count cells remain. Revisit QC before normalization.")
adata.obs["counts_before_normalization"] = counts_per_cell
adata.obs["size_factor"] = counts_per_cell / TARGET_SUM
print(adata.obs[["counts_before_normalization", "size_factor"]].describe())

In [ ]:
# Divide each cell by its library size and rescale to TARGET_SUM.
# The raw counts remain unchanged in adata.layers['counts'].
sc.pp.normalize_total(adata, target_sum=10000)
print("Median normalized total:", np.median(np.asarray(adata.X.sum(axis=1)).ravel()))

## Step 17 — Log-transform normalized expression

In [ ]:
# Apply log(1 + x) after normalization to reduce the influence of extreme expression values.
sc.pp.log1p(adata)
print("Completed log1p transformation.")

In [ ]:
# Preserve normalized/log-transformed full-gene values for later marker-gene visualization.
adata.raw = adata
print("Saved normalized/log-transformed values in adata.raw.")

## Step 18 — Choose how many highly variable genes to use, then calculate them

In [ ]:
# Choose the number of highly variable genes for dimension reduction.
N_HVGS = 2000
print("Requested highly variable genes:", N_HVGS)

In [ ]:
# Identify highly variable genes from the preserved raw-count layer.
sc.pp.highly_variable_genes(
    adata,
    layer="counts",
    flavor="seurat_v3",
    n_top_genes=min(N_HVGS, adata.n_vars),
)
print("Highly variable genes selected:", int(adata.var["highly_variable"].sum()))

In [ ]:
# Inspect the highly variable gene result before moving to PCA.
sc.pl.highly_variable_genes(adata)

## Step 19 — Scale expression and compute PCA

In [ ]:
# Scale genes for PCA; raw counts and normalized values are still preserved in layers/raw.
sc.pp.scale(adata, max_value=10)
print("Completed scaling for PCA.")

In [ ]:
# Choose how many principal components to calculate for the initial PCA diagnostic.
N_PCS_TO_COMPUTE = 50
print("Requested PCA components:", N_PCS_TO_COMPUTE)

In [ ]:
# Calculate PCA using highly variable genes.
max_possible_pcs = min(adata.n_obs - 1, int(adata.var["highly_variable"].sum()) - 1)
n_pcs_used = min(N_PCS_TO_COMPUTE, max_possible_pcs)
if n_pcs_used < 2:
    raise ValueError("Too few retained cells or highly variable genes for PCA.")
sc.tl.pca(adata, n_comps=n_pcs_used, use_highly_variable=True, svd_solver="arpack", random_state=RANDOM_SEED)
print("PCA components calculated:", n_pcs_used)

In [ ]:
# Inspect the PCA elbow plot before choosing the number of PCs for the neighbor graph.
sc.pl.pca_variance_ratio(adata, n_pcs=min(n_pcs_used, 50), log=True)

In [ ]:
# View the first two PCs colored by sample to look for technical/sample structure.
sc.pl.pca(adata, color="sample")

## Step 20 — Choose graph settings after reviewing PCA

In [ ]:
# Choose PCs for the neighbor graph after reviewing the PCA diagnostic.
N_PCS_FOR_GRAPH = 30
n_pcs_for_graph_used = min(N_PCS_FOR_GRAPH, adata.obsm["X_pca"].shape[1])
print("PCs used for graph:", n_pcs_for_graph_used)

In [ ]:
# Choose the number of nearest neighbors for graph construction.
N_NEIGHBORS = 15
print("Neighbors used for graph:", N_NEIGHBORS)

In [ ]:
# Build the k-nearest-neighbor graph from the chosen PCs.
sc.pp.neighbors(adata, n_neighbors=N_NEIGHBORS, n_pcs=n_pcs_for_graph_used, random_state=RANDOM_SEED)
print("Neighbor graph complete.")

## Step 21 — Calculate and inspect UMAP

In [ ]:
# Calculate a UMAP visualization of the neighbor graph.
sc.tl.umap(adata, random_state=RANDOM_SEED)

In [ ]:
# Inspect UMAP by sample before interpreting clusters.
sc.pl.umap(adata, color="sample")

## Step 22 — Choose and run Louvain clustering

In [ ]:
# Choose a Louvain resolution after reviewing the graph/UMAP. Larger values usually give more clusters.
LOUVAIN_RESOLUTION = 0.8
print("Louvain resolution:", LOUVAIN_RESOLUTION)

In [ ]:
# Cluster the neighbor graph with Louvain using Scanpy's igraph implementation.
sc.tl.louvain(adata, resolution=LOUVAIN_RESOLUTION, random_state=RANDOM_SEED, flavor="igraph", key_added="louvain")
print("Louvain clusters:", adata.obs["louvain"].nunique())

In [ ]:
# Inspect Louvain labels on the same UMAP coordinates.
sc.pl.umap(adata, color="louvain")

## Step 23 — Choose and run Leiden clustering

In [ ]:
# Choose a Leiden resolution independently; do not assume it must equal the Louvain resolution.
LEIDEN_RESOLUTION = 0.8
print("Leiden resolution:", LEIDEN_RESOLUTION)

In [ ]:
# Cluster the same neighbor graph with Leiden.
sc.tl.leiden(adata, resolution=LEIDEN_RESOLUTION, random_state=RANDOM_SEED, key_added="leiden")
print("Leiden clusters:", adata.obs["leiden"].nunique())

In [ ]:
# Inspect Leiden labels on UMAP.
sc.pl.umap(adata, color="leiden")

## Step 24 — Choose and run k-means clustering

In [ ]:
# Choose the number of k-means groups after inspecting the embedding and graph-based results.
KMEANS_N_CLUSTERS = 8
print("K-means clusters requested:", KMEANS_N_CLUSTERS)

In [ ]:
# Run k-means on the selected PCA coordinates. Unlike Louvain/Leiden, k-means does not use the graph.
if KMEANS_N_CLUSTERS > adata.n_obs:
    raise ValueError("KMEANS_N_CLUSTERS cannot exceed the number of retained cells.")
kmeans = KMeans(n_clusters=KMEANS_N_CLUSTERS, n_init=20, random_state=RANDOM_SEED)
adata.obs["kmeans"] = pd.Categorical(kmeans.fit_predict(adata.obsm["X_pca"][:, :n_pcs_for_graph_used]).astype(str))
print("K-means clusters:", adata.obs["kmeans"].nunique())

In [ ]:
# Inspect k-means labels on UMAP.
sc.pl.umap(adata, color="kmeans")

## Step 25 — Compare clustering labels without treating any one result as truth

In [ ]:
# Show all three clustering results on the same UMAP coordinates.
sc.pl.umap(adata, color=["louvain", "leiden", "kmeans"], wspace=0.45)

In [ ]:
# Quantify agreement between the two graph-based methods.
comparison = pd.crosstab(adata.obs["leiden"], adata.obs["louvain"], normalize="index").round(2)
display(comparison)

## Step 26 — Save the object and its decisions

Before saving, make sure the listed cutoffs reflect your actual decisions. The parameter record is part of the scientific result: it allows a future reader to reproduce what you did.

In [ ]:
# Save every important decision inside the AnnData object.
adata.uns["studio1_parameters"] = {
    "input_files": {sample: str(path) for sample, path in INPUT_FILES.items()},
    "qc_thresholds": {
        "min_genes_per_cell": MIN_GENES_PER_CELL,
        "max_genes_per_cell": MAX_GENES_PER_CELL,
        "max_mito_percent": MAX_MITO_PERCENT,
        "max_ribo_percent": MAX_RIBO_PERCENT,
        "max_hb_percent": MAX_HB_PERCENT,
        "max_total_counts": MAX_TOTAL_COUNTS,
        "min_cells_per_gene": MIN_CELLS_PER_GENE,
    },
    "expected_doublet_rate": EXPECTED_DOUBLET_RATE,
    "doublet_rule": doublet_rule,
    "doublet_candidates_removed": REMOVE_DOUBLET_CANDIDATES,
    "target_sum": TARGET_SUM,
    "n_hvgs_requested": N_HVGS,
    "n_pcs_used": n_pcs_used,
    "n_pcs_for_graph": n_pcs_for_graph_used,
    "n_neighbors": N_NEIGHBORS,
    "louvain_resolution": LOUVAIN_RESOLUTION,
    "leiden_resolution": LEIDEN_RESOLUTION,
    "kmeans_n_clusters": KMEANS_N_CLUSTERS,
    "random_seed": RANDOM_SEED,
}
print("Saved analysis decisions in adata.uns['studio1_parameters'].")

In [ ]:
# Create the results folder if needed and write the reproducible AnnData object.
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
adata.write_h5ad(OUTPUT_PATH)
print(f"Saved: {OUTPUT_PATH.resolve()}")

## Final interpretation prompt

Before calling a cluster a cell type, ask:

1. Did my QC and doublet choices change the result substantially?
2. Are the same broad groups visible under reasonable changes to PCs, neighbors, and resolution?
3. Do marker genes, sample composition, and biological replication support an interpretation?
4. What conclusion should I **not** make from this exploratory UMAP or clustering alone?

> UMAP separation and a cluster label are exploratory summaries. They are not, by themselves, evidence of a cell type or mechanism.

## References

- [Scanpy preprocessing documentation](https://scanpy.readthedocs.io/en/stable/api/preprocessing.html)
- [Scanpy Louvain documentation](https://scanpy.readthedocs.io/en/stable/api/scanpy.tl.louvain.html)
- [Scanpy Leiden documentation](https://scanpy.readthedocs.io/en/stable/api/scanpy.tl.leiden.html)
- Wolock, S. L. *et al.* (2019). Scrublet: computational identification of cell doublets in single-cell transcriptomic data. *Cell Systems*.
- Traag, V. A. *et al.* (2019). From Louvain to Leiden: guaranteeing well-connected communities. *Scientific Reports*.